In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
import numpy as np
import scipy.io
import os


@dataclass
class Trial:
    trialoutcome: str
    reactiontimes: Dict[str, float] = field(default_factory=dict)
    change_size: Optional[int] = None
    orientation: Optional[int] = None
    ITI: Optional[float] = None
    change_time: Optional[float] = None
    baseline_values: Optional[float] = None
    # Add more fields as needed for your task

@dataclass
class Cluster:
    cluster_id: int
    spike_times: np.ndarray
    quality: Optional[str] = None
    # Add more fields as needed (e.g., waveform, location, etc.)

@dataclass
class Session:
    trials: List[Trial]
    clusters: List[Cluster]
    subject: str
    session_name: str
    good_cluster_ids: Optional[List[int]] = None
    ni_events: Optional[dict] = None

def mat_struct_to_dict(obj):
    if isinstance(obj, np.ndarray):
        if obj.dtype == 'O':
            return [mat_struct_to_dict(o) for o in obj]
        else:
            return obj
    elif hasattr(obj, '_fieldnames'):
        return {field: mat_struct_to_dict(getattr(obj, field)) for field in obj._fieldnames}
    else:
        return obj

def load_mat_file_to_session(mat_path: str) -> Session:
    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    data = mat_struct_to_dict(data)
    if 'data' in data:
        data_dict = mat_struct_to_dict(data['data'])
    elif 'ans' in data:
        data_dict = mat_struct_to_dict(data['ans'])
    else:
        data_dict = data
    session_keys = list(data_dict.keys())
    session_key = session_keys[0] if session_keys else None
    subkey = list(data_dict[session_key].keys())[0] if session_key else None
    session_data = data_dict[session_key][subkey] if session_key and subkey else data_dict

    behav_data = session_data['behav_data']
    trials_raw = behav_data['trials_data_exp']
    trials = []
    for t in trials_raw:
        trial = Trial(
            trialoutcome=t.get('trialoutcome', ''),
            reactiontimes=t.get('reactiontimes', {}),
            change_size=t.get('Stim2TF', None),
            orientation=t.get('Stim2Ori', None),
            ITI = t.get('stimD', None),
            change_time=t.get('stimT', None),
            baseline_values=t.get('St1TrialVector', None)


            
        )
        trials.append(trial)
    npx_probes = session_data['NPX_probes']
    cluster_ids = np.unique(npx_probes['clu'])
    clusters = []
    for clu in cluster_ids:
        spike_times = np.array(npx_probes['st'])[np.array(npx_probes['clu']) == clu]
        cluster = Cluster(
            cluster_id=int(clu),
            spike_times=spike_times,
            quality=None
        )
        clusters.append(cluster)
    good_cluster_ids = None
    if 'cluster_id_KS_good' in npx_probes:
        good_ids = npx_probes['cluster_id_KS_good']
        if isinstance(good_ids, (np.ndarray, list)):
            good_cluster_ids = [int(x) for x in np.array(good_ids).flatten()]
        else:
            good_cluster_ids = [int(good_ids)]
    ni_events = session_data.get('NI_events', None)
    # Extract subject and session_name from NI_events['session_name']
    session_name_str = ni_events.get('session_name', 'unknown') if ni_events else 'unknown'
    parts = session_name_str.split('_')
    subject = '_'.join(parts[:2]) if len(parts) >= 3 else session_name_str
    session_name = parts[2] if len(parts) >= 3 else 'unknown'
    return Session(
        trials=trials,
        clusters=clusters,
        subject=subject,
        session_name=session_name,
        good_cluster_ids=good_cluster_ids,
        ni_events=ni_events
    )
    
# Example usage:
# session = load_mat_file_to_session('your_file.mat')

In [2]:
session = load_mat_file_to_session('E:/python_analysis/git_repos/vis_detect_analysis_Apr2023/matlab_files/BG_031_260325.mat')

In [6]:
#print and example for each field
print(f'subject name: {session.subject}')  # e.g., 'BG_031'
print(f'session name: {session.session_name}')  # e.g., '260325'
print(f'number of trials: {len(session.trials)}')  # e.g., 100
print(f'trial outcome: {session.trials[0].trialoutcome}')  # e.g., 'correct'
print(f'cluster id: {session.clusters[0].cluster_id}')  # e.g., 1    
print(f'good cluster ids: {session.good_cluster_ids}')  # e.g., [1, 2, 3]
print(f'ni events: {session.ni_events}')  # e.g., {'session_name': 'BG_031_260325', ...}
print(f'change size: {session.trials[0].change_size}')  # e.g., 2
print(f'change time: {session.trials[0].change_time}')  # e.g., 0.5
print(f'orientation: {session.trials[0].orientation}')  # e.g., 90
print(f'ITI: {session.trials[0].ITI}')  # e.g., 1.5
print(f'baseline values: {session.trials[0].baseline_values}')  # e.g., [0.1, 0.2, 0.3]
print(f'reaction times: {session.trials[0].reactiontimes}')  # e.g., {'stimulus1': 0.3, 'stimulus2': 0.4}
print(f'spike times: {session.clusters[0].spike_times}')  # e.g., array of spike times





subject name: BG_031
session name: 260325
number of trials: 925
trial outcome: abort
cluster id: 0
good cluster ids: [1, 2, 3, 6, 10, 11, 12, 13, 14, 18, 21, 22, 24, 28, 30, 32, 33, 34, 36, 39, 42, 45, 48, 50, 51, 53, 60, 64, 67, 68, 69, 72, 74, 76, 78, 79, 80, 83, 84, 85, 86, 92, 93, 95, 98, 105, 107, 111, 114, 115, 116, 117, 119, 120, 121, 123, 124, 128, 132, 133, 139, 143, 147, 149, 150, 151, 153, 154, 157, 158, 159, 161, 167, 169, 170, 173, 177, 178, 179, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 197, 198, 199, 200, 201, 205, 206, 208, 211, 212, 213, 214, 215, 217, 219, 222, 223, 224, 226, 227, 228, 230, 231, 232, 233, 235, 237, 240, 241, 247, 248, 250, 253, 256, 257, 258, 259, 264, 265, 266, 267, 268, 269, 270, 271, 272, 279, 284, 286, 287, 289, 290, 291, 293, 297, 305, 307, 311, 313, 314, 316, 330, 332, 334, 335, 336, 337, 338, 340, 341, 342, 348, 349, 357, 358, 359, 361, 362, 363, 364, 367, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382]
ni eve

In [5]:

# New cell: append this to the notebook and run it (uses mat_struct_to_dict defined above if present)
import json
import math
from collections.abc import Mapping, Sequence
from pathlib import Path
import numpy as np
import scipy.io

def _primitive(v):
    if v is None: return None
    if isinstance(v, (str, bool)): return v
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)):
        val = float(v)
        return val if math.isfinite(val) else str(v)
    return None

def summarize_array(a, n_sample=8):
    try:
        arr = np.asarray(a)
    except Exception:
        return {"type": type(a).__name__, "repr": str(a)}
    info = {"type": "ndarray", "dtype": str(arr.dtype), "shape": list(arr.shape), "size": int(arr.size)}
    if np.issubdtype(arr.dtype, np.number) and arr.size>0:
        flat = arr.flatten()
        finite = flat[np.isfinite(flat)]
        if finite.size>0:
            info.update({"min": float(np.min(finite)), "max": float(np.max(finite)),
                         "mean": float(np.mean(finite)), "std": float(np.std(finite))})
    flat = arr.flatten()
    samples = []
    for v in flat[:n_sample]:
        pv = _primitive(v)
        samples.append(pv if pv is not None else str(v))
    info["samples"] = samples
    return info

def describe(obj, max_depth=6, n_sample=6, _depth=0):
    if _depth > max_depth:
        return {"type": type(obj).__name__, "note": "max depth reached"}
    # use existing mat_struct_to_dict if available to normalize mat structs
    if hasattr(obj, '_fieldnames'):
        try:
            obj = mat_struct_to_dict(obj)  # uses notebook function if present
        except Exception:
            obj = obj
    if isinstance(obj, np.ndarray):
        return summarize_array(obj, n_sample=n_sample)
    if isinstance(obj, Mapping):
        out = {"type": "dict", "n_keys": len(obj)}
        sample_keys = list(obj.keys())[:n_sample]
        out["keys_sample"] = {}
        for k in sample_keys:
            try:
                out["keys_sample"][str(k)] = describe(obj[k], max_depth, n_sample, _depth+1)
            except Exception as e:
                out["keys_sample"][str(k)] = {"type": "error", "repr": str(e)}
        if len(obj) <= 50:
            out["all_keys"] = list(obj.keys())
        return out
    if isinstance(obj, Sequence) and not isinstance(obj, (str, bytes, bytearray)):
        length = len(obj)
        out = {"type": type(obj).__name__, "length": length}
        samples = []
        for v in list(obj)[:n_sample]:
            pv = _primitive(v)
            samples.append(pv if pv is not None else describe(v, max_depth, n_sample, _depth+1))
        out["samples"] = samples
        return out
    pv = _primitive(obj)
    if pv is not None:
        return {"type": type(obj).__name__, "value": pv}
    return {"type": type(obj).__name__, "repr": str(obj)}

def find_first_session_like(data_dict):
    """
    Improved heuristic:
    - Prefer the 'data' top-level key (MATLAB saves main variables there).
    - Skip metadata keys (__header__, __version__, __globals__).
    - Look for dicts that contain any of ('behav_data','NPX_probes','NI_events','trials_data_exp').
    - Return a string key path and the corresponding dict/struct converted with mat_struct_to_dict.
    """
    if not isinstance(data_dict, dict):
        return None, data_dict

    # 1) Prefer 'data' key if present
    if 'data' in data_dict:
        try:
            data_branch = mat_struct_to_dict(data_dict['data']) if not isinstance(data_dict['data'], dict) else data_dict['data']
        except Exception:
            data_branch = data_dict['data']
        if isinstance(data_branch, dict) and data_branch:
            # search entries under data/
            for k, v in data_branch.items():
                try:
                    d = mat_struct_to_dict(v) if not isinstance(v, dict) else v
                except Exception:
                    d = v
                if isinstance(d, Mapping) and any(x in d for x in ('behav_data', 'NPX_probes', 'NI_events', 'trials_data_exp')):
                    return f"data/{k}", d
            # fallback: return first nested key under data
            first_key = next(iter(data_branch.keys()))
            try:
                return f"data/{first_key}", mat_struct_to_dict(data_branch[first_key])
            except Exception:
                return f"data/{first_key}", data_branch[first_key]

    # 2) General scan of top-level keys, skipping MATLAB metadata
    for k, v in data_dict.items():
        if k in ('__header__', '__version__', '__globals__'):
            continue
        try:
            d = mat_struct_to_dict(v) if not isinstance(v, dict) else v
        except Exception:
            d = v
        if isinstance(d, Mapping):
            if any(x in d for x in ('behav_data', 'NPX_probes', 'NI_events', 'trials_data_exp')):
                return k, d
            # check one level deeper
            for k2, v2 in d.items():
                try:
                    d2 = mat_struct_to_dict(v2) if not isinstance(v2, dict) else v2
                except Exception:
                    d2 = v2
                if isinstance(d2, Mapping) and any(x in d2 for x in ('behav_data', 'NPX_probes', 'NI_events', 'trials_data_exp')):
                    return f"{k}/{k2}", d2

    # 3) fallback: pick the first non-metadata key
    for k in data_dict.keys():
        if k not in ('__header__', '__version__', '__globals__'):
            try:
                return k, mat_struct_to_dict(data_dict[k])
            except Exception:
                return k, data_dict[k]

    # final fallback (shouldn't normally be reached)
    first_key = next(iter(data_dict.keys()))
    return first_key, mat_struct_to_dict(data_dict[first_key])

def extract_examples_from_session_obj(session_obj, n_spikes=100):
    # tries common locations for examples: first trial, first cluster, NI_events
    examples = {}
    # first trial
    trials = None
    if isinstance(session_obj, dict):
        trials = session_obj.get('behav_data', {}).get('trials_data_exp') if isinstance(session_obj.get('behav_data'), dict) else session_obj.get('trials_data_exp') or session_obj.get('trials')
    if isinstance(trials, (list, np.ndarray)) and len(trials)>0:
        examples['first_trial'] = trials[0]
    # NPX probes => clusters
    np_probes = session_obj.get('NPX_probes') if isinstance(session_obj, dict) else None
    if isinstance(np_probes, dict) and 'st' in np_probes and 'clu' in np_probes:
        try:
            st = np.asarray(np_probes['st'])
            clu = np.asarray(np_probes['clu'])
            unique = np.unique(clu)
            if unique.size>0:
                first_clu = unique[0]
                spikes = st[clu==first_clu]
                examples['first_cluster_id'] = int(first_clu)
                examples['first_cluster_spike_times_sample'] = spikes[:n_spikes].tolist()
        except Exception:
            pass
    # NI_events
    if isinstance(session_obj, dict) and 'NI_events' in session_obj:
        examples['NI_events'] = session_obj['NI_events']
    return examples

def generate_and_append_raw_schema(mat_path, out_md_path=None, out_json_path=None, repo_docs_path=None):
    mat_path = Path(mat_path)
    if out_md_path is None:
        repo_root = mat_path.parents[2] if len(mat_path.parents) > 2 else mat_path.parent
        docs_dir = repo_root / "docs"
        docs_dir.mkdir(exist_ok=True)
        out_md_path = docs_dir / "SESSION_SCHEMA.md"
    if out_json_path is None:
        out_json_path = mat_path.parent / f"RAW_SESSION_SCHEMA_{mat_path.stem}.json"

    raw = scipy.io.loadmat(str(mat_path), struct_as_record=False, squeeze_me=True)
    # convert MATLAB structs to python-native using mat_struct_to_dict if present
    raw_conv = mat_struct_to_dict(raw) if 'mat_struct_to_dict' in globals() else raw
    top_keys = list(raw_conv.keys()) if isinstance(raw_conv, dict) else []

    session_key, session_obj = find_first_session_like(raw_conv)

    summary = {
        "mat_file": str(mat_path),
        "top_level_keys": {},
        "selected_session_key": session_key,
        "session_description": None
    }

    for k in top_keys:
        v = raw_conv[k]
        try:
            summary["top_level_keys"][k] = describe(v, max_depth=1, n_sample=4)
        except Exception as e:
            summary["top_level_keys"][k] = {"type": "error", "repr": str(e)}

    summary["session_description"] = describe(session_obj, max_depth=6, n_sample=8)

    # write JSON summary (use default=str to handle arrays)
    with open(out_json_path, "w", encoding="utf8") as f:
        json.dump(summary, f, indent=2, default=str)

    # append examples + summary to docs/SESSION_SCHEMA.md
    md_lines = []
    md_lines.append(f"## Raw session schema for {mat_path.name}")
    md_lines.append(f"- Mat file: `{mat_path}`")
    md_lines.append(f"- Detected session key: `{session_key}`")
    md_lines.append("")
    md_lines.append("### Top-level keys (short)")
    for k, v in summary["top_level_keys"].items():
        md_lines.append(f"#### {k}")
        md_lines.append("```json")
        md_lines.append(json.dumps(v, indent=2, default=str))
        md_lines.append("```")
        md_lines.append("")
    md_lines.append("### Selected session object (detailed JSON)")
    md_lines.append("```json")
    md_lines.append(json.dumps(summary["session_description"], indent=2, default=str))
    md_lines.append("```")
    md_lines.append("")

    examples = extract_examples_from_session_obj(session_obj, n_spikes=200)
    if examples:
        md_lines.append("### Example raw content (first trial / first cluster / NI_events if present)")
        md_lines.append("```json")
        md_lines.append(json.dumps(examples, indent=2, default=str))
        md_lines.append("```")
        md_lines.append("")

    # append to file
    out_md_path = Path(out_md_path)
    out_md_path.parent.mkdir(parents=True, exist_ok=True)
    with out_md_path.open("a", encoding="utf8") as fh:
        fh.write("\n".join(md_lines))
        fh.write("\n\n<!-- appended by generate_and_append_raw_schema -->\n")

    return {"md": str(out_md_path), "json": str(out_json_path)}

# Example usage in your notebook:
# run this cell, then:
# res = generate_and_append_raw_schema(r"E:\python_analysis\git_repos\vis_detect_analysis_Apr2023\matlab_files\BG_031_260325.mat")
# print("Wrote:", res)

In [6]:
res = generate_and_append_raw_schema(r"E:\python_analysis\git_repos\vis_detect_analysis_Apr2023\matlab_files\BG_031_260325.mat")